In [6]:
from pyspark.sql import SparkSession
from delta.tables import DeltaTable
!pwd
batch_table_path = "/opt/workspace/delta-trades-batch-table"
stream_table_path = "/opt/workspace/delta-trades-stream-5s-table"

/opt/workspace


## 1. Table description

In [5]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("DeltaInspect") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

# 1️⃣ ✅ SCHEMA + PARTITIONING + LOCATION + etc.
print("=== TABLE STRUCTURE (DESCRIBE DETAIL) ===")
spark.sql(f"DESCRIBE DETAIL delta.`{stream_table_path}`").show(truncate=True)

# 2️⃣ ✅ WRITE HISTORY (includes file count, row count, size)
print("\n=== LAST WRITE METRICS (DESCRIBE HISTORY) ===")
spark.sql(f"DESCRIBE HISTORY delta.`{stream_table_path}`").select(
    "version", "timestamp", "operation", "operationMetrics"
).show(truncate=False)

# 3️⃣ ✅ RAW SCHEMA (if you just want column names/types)
print("\n=== SCHEMA ONLY ===")
spark.sql(f"DESCRIBE delta.`{stream_table_path}`").show(30)

=== TABLE STRUCTURE (DESCRIBE DETAIL) ===


+------+--------------------+----+-----------+--------------------+--------------------+--------------------+----------------+-----------------+--------+-----------+----------+----------------+----------------+--------------------+
|format|                  id|name|description|            location|           createdAt|        lastModified|partitionColumns|clusteringColumns|numFiles|sizeInBytes|properties|minReaderVersion|minWriterVersion|       tableFeatures|
+------+--------------------+----+-----------+--------------------+--------------------+--------------------+----------------+-----------------+--------+-----------+----------+----------------+----------------+--------------------+
| delta|b3a8ebe7-9a4d-406...|NULL|       NULL|file:/opt/workspa...|2025-12-29 09:24:...|2025-12-29 09:36:...|       [hour_id]|               []|     621|    3977682|        {}|               1|               2|[appendOnly, inva...|
+------+--------------------+----+-----------+--------------------+-----

## 2. Table query

### 2.1. Batch table query

In [16]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("DeltaRead") \
    .config("spark.ui.port", "4041") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.sql.catalogImplementation", "in-memory") \
    .getOrCreate()

# Now this should work in OSS Delta Lake 3.2.0 + Spark 3.5.1
files_df = spark.sql(f"SELECT * FROM delta.`{batch_table_path}`")
print('Partitions from reading delta lake: ', files_df.rdd.getNumPartitions())
files_df.createOrReplaceTempView("batch_trades")
# files_df.show(5, truncate=True)

25/12/29 08:47:08 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


Partitions from reading delta lake:  12


In [17]:
df_batch_check = spark.sql(f"""
    SELECT
    exchange_minute, exchange_hour, count(distinct trade_id) as trade_cnt
    FROM batch_trades
    where exchange_day = 29
    group by 1, 2
    order by 2 desc, 1 desc
""").show(20)

+---------------+-------------+---------+
|exchange_minute|exchange_hour|trade_cnt|
+---------------+-------------+---------+
|             29|           14|      237|
|             28|           14|      696|
|             27|           14|      619|
|             26|           14|      699|
|             25|           14|      730|
|             24|           14|      999|
|             23|           14|      680|
|             22|           14|      869|
|             21|           14|     1348|
|             20|           14|      972|
|             19|           14|     1129|
|             18|           14|      790|
|             17|           14|      893|
|             16|           14|     1677|
|             15|           14|      494|
|             14|           14|      657|
|             13|           14|     1188|
|             12|           14|     1025|
|             11|           14|     1009|
|             10|           14|     1438|
+---------------+-------------+---

### 2.2. Stream table query

#### 2.2.1. Read stream table

In [17]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("DeltaRead") \
    .config("spark.ui.port", "4043") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.sql.catalogImplementation", "in-memory") \
    .getOrCreate()

df_stream = spark.sql(f"SELECT * FROM delta.`{stream_table_path}`")
print('Partitions from reading df_stream: ', df_stream.rdd.getNumPartitions())
df_stream.createOrReplaceTempView("stream_trades")
df_stream.show(5, truncate=True)

25/12/29 09:42:18 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.
                                                                                

Partitions from reading df_stream:  31


+------+-------+-------+---------+----+------+----------+--------------------+--------------------+--------------------+--------------------+-------------------+---------------+---------------+-------------+------------+--------------+----------+
|offset|    key| symbol| trade_id|side| price|      size|       exchange_time|         ingest_time|    exchange_time_ts|      ingest_time_ts| ingest_time_minute|exchange_second|exchange_minute|exchange_hour|exchange_day|exchange_month|   hour_id|
+------+-------+-------+---------+----+------+----------+--------------------+--------------------+--------------------+--------------------+-------------------+---------------+---------------+-------------+------------+--------------+----------+
|369943|XRP-USD|XRP-USD|202754575| buy| 1.892|     787.5|2025-12-29T09:27:...|2025-12-29T09:27:...|2025-12-29 16:27:...|2025-12-29 16:27:...|2025-12-29 16:27:00|             12|             27|           16|          29|            12|2025122916|
|369944|XRP-

#### 2.2.2. Checking data quality

In [16]:
spark.sql(f"""
    SELECT
    exchange_minute, exchange_hour, count(distinct trade_id) as trade_cnt
    FROM stream_trades
    group by 1, 2
    order by 2 desc, 1 desc
""").show()

[Stage 74:================================================>       (25 + 4) / 29]

+---------------+-------------+---------+
|exchange_minute|exchange_hour|trade_cnt|
+---------------+-------------+---------+
|             41|           16|     1406|
|             40|           16|     1605|
|             39|           16|     1174|
|             38|           16|     1508|
|             37|           16|     2583|
|             36|           16|     1841|
|             35|           16|     1545|
|             34|           16|     1499|
|             33|           16|     2505|
|             32|           16|     2242|
|             31|           16|     3825|
|             30|           16|      573|
|             29|           16|      244|
|             28|           16|      395|
|             27|           16|     1106|
|             26|           16|     1882|
|             25|           16|      894|
|             24|           16|       92|
+---------------+-------------+---------+



#### 2.2.3. Data transformation

In [24]:
df_5min = spark.sql("""
    SELECT
        window.start AS window_start,
        window.end AS window_end,
        symbol,
        COUNT(*) AS trade_count,
        SUM(case when side = 'buy' then size end) AS volume_buy,
        SUM(case when side = 'sell' then size end) AS volume_sell,
        AVG(price) AS avg_price,
        MIN(price) AS min_price,
        MAX(price) AS max_price,
        LAST(price, TRUE) AS close_price,  -- last trade price in window
        FIRST(price, TRUE) AS open_price   -- first trade price in window
    FROM (
        SELECT *,
               window(exchange_time_ts, '5 minutes') AS window
        FROM stream_trades
    )
    GROUP BY window, symbol
    ORDER BY window_start DESC, symbol
""")

df_5min.printSchema()
df_5min.show(10, truncate=False)

root
 |-- window_start: timestamp (nullable = true)
 |-- window_end: timestamp (nullable = true)
 |-- symbol: string (nullable = true)
 |-- trade_count: long (nullable = false)
 |-- volume_buy: double (nullable = true)
 |-- volume_sell: double (nullable = true)
 |-- avg_price: double (nullable = true)
 |-- min_price: double (nullable = true)
 |-- max_price: double (nullable = true)
 |-- close_price: double (nullable = true)
 |-- open_price: double (nullable = true)



[Stage 154:==================================================>    (37 + 3) / 40]

+-------------------+-------------------+-------+-----------+------------------+------------------+-------------------+---------+---------+-----------+----------+
|window_start       |window_end         |symbol |trade_count|volume_buy        |volume_sell       |avg_price          |min_price|max_price|close_price|open_price|
+-------------------+-------------------+-------+-----------+------------------+------------------+-------------------+---------+---------+-----------+----------+
|2025-12-29 16:45:00|2025-12-29 16:50:00|ADA-USD|364        |162364.71059076997|109395.70658647   |0.36769670329670345|0.3669   |0.3687   |0.3676     |0.3673    |
|2025-12-29 16:45:00|2025-12-29 16:50:00|BTC-USD|3482       |40.28869031000001 |10.846198720000007|87960.60836588149  |87869.59 |88134.36 |87921.37   |88020.0   |
|2025-12-29 16:45:00|2025-12-29 16:50:00|ETH-USD|1360       |210.04045015000008|170.71252654000006|2959.5825882352938 |2953.72  |2967.72  |2959.32    |2963.21   |
|2025-12-29 16:45:00|2

## 3. Time travel

In [11]:
# Read the table as it existed at a specific time
df_past = spark.sql(f"""
    SELECT * FROM delta.`{table_path}`
    TIMESTAMP AS OF '2025-12-28 19:00:00'
""")
df_past.count()

1724793

In [14]:
# Read the table as it existed at a specific time
df_current = spark.sql(f"""
    SELECT * FROM delta.`{table_path}`
""")
df_current.count()

2298468